In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchsummary import summary

In [2]:
class VGG16(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Block 1: 2层, 64通道
        self.b1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # Block 2: 2层, 128通道 
        self.b2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # Block 3: 3层, 256通道
        self.b3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # Block 4: 3层, 512通道
        self.b4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # Block 5: 3层, 512通道
        self.b5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 全连接层 (Classifier)
        # 将FC层放入Sequential中，包含ReLU和Dropout
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5), # VGG标配Dropout
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.b1(x)
        x = self.pool(x)
        
        x = self.b2(x)
        x = self.pool(x)
        
        x = self.b3(x)
        x = self.pool(x)
        
        x = self.b4(x)
        x = self.pool(x)
        
        x = self.b5(x)
        x = self.pool(x)
        
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [3]:
model = VGG16()
model.cuda()
summary(model, input_size=(3, 224, 224))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
            Conv2d-3         [-1, 64, 224, 224]          36,928
              ReLU-4         [-1, 64, 224, 224]               0
         MaxPool2d-5         [-1, 64, 112, 112]               0
            Conv2d-6        [-1, 128, 112, 112]          73,856
              ReLU-7        [-1, 128, 112, 112]               0
            Conv2d-8        [-1, 128, 112, 112]         147,584
              ReLU-9        [-1, 128, 112, 112]               0
        MaxPool2d-10          [-1, 128, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]         295,168
             ReLU-12          [-1, 256, 56, 56]               0
           Conv2d-13          [-1, 256, 56, 56]         590,080
             ReLU-14          [-1, 256,

In [4]:
import torch.utils.data as Data
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.datasets import FashionMNIST

# 数据集构建
full_data = FashionMNIST(root='./data',train=True,transform=transforms.Compose([transforms.Resize(size=224),transforms.Grayscale(num_output_channels=3),transforms.ToTensor()]),download=True)#这里是1通道的灰度图
# 划分训练集和验证集 
train_data,val_data = Data.random_split(dataset=full_data,lengths=[round(0.8*len(full_data)),round(0.2*len(full_data))])
train_loader = Data.DataLoader(dataset=train_data,batch_size=256,shuffle=True,num_workers=0)
val_loader = Data.DataLoader(dataset=val_data,batch_size=256,shuffle=False,num_workers=0)
# 获得一个batch的数据
i=1
for batch_idx,(data,target) in enumerate(train_loader):
    if i>0:
      print(data.shape)
      print(target.shape)
      i-=1
    if batch_idx > 0:
      break
    batch_x = data.squeeze().numpy()#将四维张亮去掉第一维
    batch_y=target.numpy()
    class_label = train_data.dataset.classes#训练集的类别标签
    # 打印数据形状    
    print(batch_x.shape)
    print(batch_y.shape)
    print(class_label)


torch.Size([256, 3, 224, 224])
torch.Size([256])
(256, 3, 224, 224)
(256,)
['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [5]:
# 定义优化器和损失函数
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
criterion =nn.CrossEntropyLoss()#交叉熵用于分类

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import wandb
import copy
import math


physical_batch_size = 1
# 目标等效 batch_size = 64
# 那么 accumulation_steps = 64 / 1 = 64
accumulation_steps = 64
epoch_num = 100
warmup_epochs = 5
lr = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= 准备工作 =================
# 1. 定义模型、优化器、损失函数
model = VGG16().to(device)
optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4) # VGG常用SGD
criterion = nn.CrossEntropyLoss()

# 2. 定义调度器 (Warmup + Cosine)
# 阶段1: Warmup (从 lr/100 线性增加到 lr)
scheduler_warmup = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs)
# 阶段2: Cosine Annealing (从 lr 降到 0)
scheduler_cosine = CosineAnnealingLR(optimizer, T_max=epoch_num - warmup_epochs)
# 组合在一起
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs])

# 3. 初始化 AMP Scaler
scaler = torch.amp.GradScaler('cuda')

# 4. WandB 初始化
wandb.init(
    project="VGG16_3050Ti_Optimization",
    config={
        "lr": lr,
        "epochs": epoch_num,
        "batch_size_effective": physical_batch_size * accumulation_steps,
        "physical_batch_size": physical_batch_size,
        "accumulation_steps": accumulation_steps,
        "scheduler": "Warmup+Cosine",
        "amp": True
    }
)

# 训练记录
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

# ================= 主循环 =================
main_loop = tqdm(range(epoch_num), desc="Training", unit='epoch')

for epoch in main_loop:
    train_loss = 0.0
    train_acc = 0.0
    val_loss = 0.0
    val_acc = 0.0
    
    # === 训练阶段 ===
    model.train()
    optimizer.zero_grad() # 循环开始前清零一次
    
    # tqdm 显示步数
    train_steps = len(train_loader)
    
    for step, (b_x, b_y) in enumerate(train_loader):
        b_x, b_y = b_x.to(device), b_y.to(device)

        # 1. AMP 上下文 (自动混合精度)
        with torch.amp.autocast('cuda'):
            output = model(b_x)
            loss = criterion(output, b_y)
            # 2. 梯度累计的关键：损失除以累计步数
            loss = loss / accumulation_steps

        # 3. 反向传播 (使用 scaler)
        scaler.scale(loss).backward()

        # 记录 Loss (乘回去是为了显示真实的 step loss)
        train_loss += loss.item() * accumulation_steps * b_x.size(0)
        pre_label = torch.argmax(output, 1)
        train_acc += torch.sum(pre_label == b_y.data)

        # 4. 真正更新参数的时刻
        if (step + 1) % accumulation_steps == 0 or (step + 1) == train_steps:
            # 4.1 Unscale 梯度 (为了裁剪)
            scaler.unscale_(optimizer)
            
            # 4.2 梯度裁剪 (防止爆炸，通常设为 1.0 或 5.0)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 4.3 更新参数
            scaler.step(optimizer)
            
            # 4.4 更新 Scaler
            scaler.update()
            
            # 4.5 清空梯度
            optimizer.zero_grad()

    # === 调度器更新 ===
    # 注意：CosineAnnealing 通常按 epoch 更新
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    # === 验证阶段 (也建议用 AMP 加速) ===
    model.eval()
    val_num = 0
    with torch.no_grad():
        for b_x, b_y in val_loader:
            b_x, b_y = b_x.to(device), b_y.to(device)
            
            with autocast(): # 验证也开 AMP，省显存
                output = model(b_x)
                loss = criterion(output, b_y)
            
            val_loss += loss.item() * b_x.size(0)
            pre_label = torch.argmax(output, 1)
            val_acc += torch.sum(pre_label == b_y.data)
            val_num += b_x.size(0)

    # === 指标计算 ===
    train_num = len(train_loader.dataset)
    epoch_train_loss = train_loss / train_num
    epoch_train_acc = (train_acc.double() / train_num).item()
    epoch_val_loss = val_loss / val_num
    epoch_val_acc = (val_acc.double() / val_num).item()

    # === 记录与保存 ===
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        wandb.run.summary["best_accuracy"] = best_acc

    wandb.log({
        "Train Loss": epoch_train_loss,
        "Train Acc": epoch_train_acc,
        "Val Loss": epoch_val_loss,
        "Val Acc": epoch_val_acc,
        "LR": current_lr,
        "Epoch": epoch
    })
    
    main_loop.set_postfix(
        lr=f"{current_lr:.2e}",
        t_loss=f"{epoch_train_loss:.4f}", 
        v_acc=f"{epoch_val_acc:.4f}"
    )

# 结束
wandb.finish()
torch.save(best_model_wts, 'best_model_optimized.pth')
print(f"Done. Best Acc: {best_acc}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Training:   0%|          | 0/100 [00:42<?, ?epoch/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 196.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.33 GiB is allocated by PyTorch, and 111.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)